In [1]:
import torch
import random
import math
import numpy as np

In [2]:
x1 = torch.Tensor([2.0]).double()                  ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()                  ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()                 ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()                  ; w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()    ; b.requires_grad = True

n = x1*w1+x2*w2+b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('---')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

0.7071066904050358
---
x2 0.5000001283844369
w2 0.0
x1 -1.5000003851533106
w1 1.0000002567688737


In [3]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self._grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label
        
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self,other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data+other.data, (self, other), '+')

        def _backward():
            self._grad += 1.0*out._grad
            other._grad += 1.0*out._grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self+other
        
    def __pow__(self,other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')
        
        def _backward():
            self._grad += (other*self.data**(other-1))*out._grad
        out._backward = _backward
        return out
        
    def __mul__(self,other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data*other.data, (self, other), '*')
        
        def _backward():
            self._grad += other.data*out._grad
            other._grad += self.data*out._grad
        out._backward = _backward
        return out

    def __truediv__(self, other): # self/other
        return self*other**-1

    def __rmul__(self, other):
        return self*other
        
    def __neg__(self):
        return self*-1
    def __sub__(self, other):
        return self+(-other)

    def tanh(self):
        x = self.data
        t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out = Value(t, (self,), 'tanh')
        
        def _backward():
            self._grad += (1.0-t**2)*out._grad
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')
        def _backward():
            self._grad += out.data*out._grad
        out._backward = _backward
        return out

    def backward(self):
        self._grad = 1.0
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        for node in reversed(topo):
            node._backward()

In [4]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f}" % (n.label, n.data, n._grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

## Making a Neuron

In [5]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        ## w*x + b
        a = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        out = a.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [150]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4,4,1])
n(x)

Value(data=0.6415568723960241)

In [151]:
len(n.parameters())

41

In [152]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]
ypred = [n(x) for x in xs]
ypred

[Value(data=0.6415568723960241),
 Value(data=-0.35251180237763313),
 Value(data=0.7245674901251822),
 Value(data=0.5086965408811589)]

In [153]:
for k in range(20):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout-ygt)**2 for ygt, yout in zip(ys, ypred))

    # backward
    for p in n.parameters():
        p._grad = 0.0
    loss.backward()

    # update
    for p in n.parameters():
        p.data += -0.05*p._grad

    print(f"step: {k}  | loss: {loss.data}")

step: 0  | loss: 3.7632345587255904
step: 1  | loss: 3.38293573886279
step: 2  | loss: 2.925969934388065
step: 3  | loss: 2.198830640946334
step: 4  | loss: 1.1322961059627765
step: 5  | loss: 0.26120771292104056
step: 6  | loss: 0.05586485603888487
step: 7  | loss: 0.0145850815090732
step: 8  | loss: 0.004340400897827765
step: 9  | loss: 0.00150892553823032
step: 10  | loss: 0.000601585288618359
step: 11  | loss: 0.0002645387836823311
step: 12  | loss: 0.00012376198155970305
step: 13  | loss: 6.0060838180596975e-05
step: 14  | loss: 2.977258968724953e-05
step: 15  | loss: 1.4946314164544401e-05
step: 16  | loss: 7.564262556504571e-06
step: 17  | loss: 3.850249620289156e-06
step: 18  | loss: 1.968631126949275e-06
step: 19  | loss: 1.0104081854008817e-06


In [154]:
ypred

[Value(data=0.9992897624200053),
 Value(data=-0.9999936693820268),
 Value(data=-0.999999902433514),
 Value(data=0.9992887119576985)]